## 各个SV工具输出的vcf文件格式差异很大，不利于统计，为了便于后续分析，计划在不改动原始vcf结果数据的前提下对部分BND添加注释信息，且在后续统计时优先阅读。

## （此pipeline紧接sv工具结果）

# 一、需要用到的脚本

### annotate_bnd_with_inferred_svtype.py：

In [ ]:
#!/usr/bin/env python3
"""
Annotate BND records with inferred SVTYPE (BNDINF_*). Does NOT change SVTYPE / ALT / MATEID.

- GRIDSS  : read EVENTTYPE
- DELLY   : read CT
- LUMPY   : read STRANDS
- MANTA   : infer from ALT orientation
- SvABA   : infer from ALT orientation
- CUE     : skip (no BND)

New INFO fields:
  BNDINF_SVTYPE, BNDINF_SOURCE, BNDINF_METHOD, BNDINF_CHR2, BNDINF_POS2
"""
import argparse
import gzip
import re
from collections import Counter
from pathlib import Path


BND_RE = re.compile(r"([\[\]])([^:\[\]]+):([0-9]+)([\[\]])")

CT_TO_SVTYPE = {"3to5": "DEL", "5to3": "DUP", "3to3": "INV", "5to5": "INV"}
STRANDS_TO_SVTYPE = {"+-": "DEL", "-+": "DUP", "++": "INV", "--": "INV"}
GRIDSS_PASSTHROUGH = {"DEL", "DUP", "INV", "INS", "SGL", "BND", "TRA"}

NEW_INFO_HEADERS = [
    '##INFO=<ID=BNDINF_SVTYPE,Number=1,Type=String,Description="SV type inferred for this BND record; original SVTYPE is unchanged">',
    '##INFO=<ID=BNDINF_SOURCE,Number=1,Type=String,Description="native: from tool native field; inferred: from breakend orientation">',
    '##INFO=<ID=BNDINF_METHOD,Number=1,Type=String,Description="Concrete rule used to infer BNDINF_SVTYPE">',
    '##INFO=<ID=BNDINF_CHR2,Number=1,Type=String,Description="Remote chromosome for this BND record">',
    '##INFO=<ID=BNDINF_POS2,Number=1,Type=Integer,Description="Remote position for this BND record">',
]


def open_text(path, mode):
    if str(path).endswith(".gz"):
        return gzip.open(path, mode + "t")
    return open(path, mode)


def parse_info(info):
    d = {}
    if info in {"", "."}:
        return d
    for item in info.split(";"):
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
        else:
            d[item] = True
    return d


def format_info(info):
    out = []
    for k, v in info.items():
        if v is True:
            out.append(k)
        elif v is False or v is None:
            continue
        else:
            out.append(f"{k}={v}")
    return ";".join(out) if out else "."


def infer_tool(path):
    s = str(path).lower()
    for tool in ["cue", "delly", "lumpy", "manta", "svaba", "gridss", "gripss"]:
        if tool in s:
            return "gridss" if tool == "gripss" else tool
    return "unknown"


def norm_chrom(chrom):
    chrom = str(chrom or "")
    return chrom[3:] if chrom.lower().startswith("chr") else chrom


def is_bnd_record(fields):
    if len(fields) < 8:
        return False
    info = parse_info(fields[7])
    svtype = str(info.get("SVTYPE", "")).upper()
    return svtype == "BND" or bool(BND_RE.search(fields[4]))


def parse_bnd_alt(alt):
    m = BND_RE.search(alt)
    if not m:
        return None
    bracket, rchrom, rpos, _ = m.groups()
    local_side = "5" if alt[0] in "[]" else "3"
    remote_side = "5" if bracket == "[" else "3"
    return {
        "remote_chrom": rchrom,
        "remote_pos": int(rpos),
        "local_side": local_side,
        "remote_side": remote_side,
    }

def literal_insert_len(ref, alt):
    seq = BND_RE.sub("", alt)
    seq = "".join(x for x in seq.upper() if x in "ACGTN")
    return max(0, len(seq) - len(ref))

def classify_breakend(chrom, pos, ref, alt, ins_max_distance=50, min_inserted_seq=20):
    """Generic orientation-based classifier (used for MANTA / SvABA)."""
    bnd = parse_bnd_alt(alt)
    if not bnd:
        return "BND", "unparsed_BND_ALT"

    rchrom = bnd["remote_chrom"]
    rpos = bnd["remote_pos"]
    if norm_chrom(chrom) != norm_chrom(rchrom):
        return "TRA", "different_chromosome"
    
    ins_len = literal_insert_len(ref, alt)
    if abs(pos - rpos) <= ins_max_distance and ins_len >= min_inserted_seq:
        return "INS", f"short_distance_with_inserted_sequence_len={ins_len}"
    
    if pos <= rpos:
        left_side = bnd["local_side"]
        right_side = bnd["remote_side"]
    else:
        left_side = bnd["remote_side"]
        right_side = bnd["local_side"]

    pair = left_side + "to" + right_side
    if pair == "3to5":
        return "DEL", pair
    if pair == "5to3":
        return "DUP", pair
    if pair in {"3to3", "5to5"}:
        return "INV", pair
    return "BND", "unresolved_orientation"


def infer_svtype_for_bnd(tool, chrom, pos, ref, alt, info):
    bnd = parse_bnd_alt(alt)
    chr2 = bnd["remote_chrom"] if bnd else None
    pos2 = bnd["remote_pos"] if bnd else None

    # 1. 跨染色体 -> TRA
    if bnd and norm_chrom(chrom) != norm_chrom(bnd["remote_chrom"]):
        return "TRA", "inferred", "different_chromosome", chr2, pos2

    # 2. 原生字段
    if tool == "gridss" and "EVENTTYPE" in info and info["EVENTTYPE"] is not True:
        sv = str(info["EVENTTYPE"]).upper()
        if sv in GRIDSS_PASSTHROUGH:
            return sv, "native", f"eventtype_{sv.lower()}", chr2, pos2

    if tool == "delly" and "CT" in info and info["CT"] is not True:
        raw = str(info["CT"])
        sv = CT_TO_SVTYPE.get(raw)
        if sv:
            return sv, "native", f"ct_{raw}", chr2, pos2

    if tool == "lumpy" and "STRANDS" in info and info["STRANDS"] is not True:
        strand = str(info["STRANDS"]).split(":")[0].strip()
        sv = STRANDS_TO_SVTYPE.get(strand)
        if sv:
            method = "strands_" + strand.replace("+", "plus").replace("-", "minus")
            return sv, "native", method, chr2, pos2

    # 3. 通用 orientation 推断
    sv, method = classify_breakend(chrom, pos, ref, alt)
    return sv, "inferred", method, chr2, pos2


def annotate_bnd_info(fields, tool):
    info = parse_info(fields[7])
    if "BNDINF_SVTYPE" in info:
        return fields, "already_annotated"

    chrom = fields[0]
    try:
        pos = int(float(fields[1]))
    except Exception:
        pos = 0
    ref = fields[3]
    alt = fields[4]

    svtype, source, method, chr2, pos2 = infer_svtype_for_bnd(tool, chrom, pos, ref, alt, info)

    info["BNDINF_SVTYPE"] = svtype
    info["BNDINF_SOURCE"] = source
    info["BNDINF_METHOD"] = method
    if chr2 is not None:
        info["BNDINF_CHR2"] = chr2
    if pos2 is not None:
        info["BNDINF_POS2"] = pos2

    fields[7] = format_info(info)
    return fields, f"{source}:{svtype}"


def read_records(path):
    headers, records = [], []
    with open_text(path, "r") as handle:
        for line in handle:
            if line.startswith("#"):
                headers.append(line)
            else:
                records.append(line.rstrip("\n").split("\t"))
    return headers, records


def write_records(path, headers, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    existing = set()
    for line in headers:
        m = re.match(r"##INFO=<ID=(BNDINF_\w+)", line)
        if m:
            existing.add(m.group(1))

    with open_text(path, "w") as out:
        inserted = False
        for line in headers:
            if line.startswith("#CHROM") and not inserted:
                for h in NEW_INFO_HEADERS:
                    m = re.match(r"##INFO=<ID=(BNDINF_\w+)", h)
                    if m and m.group(1) not in existing:
                        out.write(h + "\n")
                inserted = True
            out.write(line)
        for fields in records:
            out.write("\t".join(fields) + "\n")


def collect_vcf_files(inputs, include_checkpoints):
    files = []
    for item in inputs:
        p = Path(item)
        if p.is_dir():
            files.extend(sorted(p.rglob("*.vcf")))
            files.extend(sorted(p.rglob("*.vcf.gz")))
        elif p.exists():
            files.append(p)
    if not include_checkpoints:
        files = [p for p in files if ".ipynb_checkpoints" not in p.parts]
    return files


def output_path_for(infile, outdir, tool):
    name = infile.name
    if name.endswith(".vcf.gz"):
        out_name = name[:-7] + ".bnd_annotated.vcf.gz"
    elif name.endswith(".vcf"):
        out_name = name[:-4] + ".bnd_annotated.vcf"
    else:
        out_name = name + ".bnd_annotated.vcf"
    return outdir / tool / out_name


def process_vcf(infile, outdir):
    tool = infer_tool(infile)
    outfile = output_path_for(infile, outdir, tool)
    headers, records = read_records(infile)
    counts = Counter()
    annotated = []

    for fields in records:
        if len(fields) < 8:
            annotated.append(fields)
            counts["malformed"] += 1
            continue
        if not is_bnd_record(fields):
            annotated.append(fields)
            counts["non_bnd_untouched"] += 1
            continue
        fields, tag = annotate_bnd_info(fields, tool)
        annotated.append(fields)
        counts[tag] += 1

    write_records(outfile, headers, annotated)
    return tool, infile, outfile, counts


def main():
    parser = argparse.ArgumentParser(
        description="Annotate BND records with BNDINF_* (inferred SVTYPE). Does NOT change SVTYPE / ALT / MATEID."
    )
    parser.add_argument("-i", "--input", nargs="+", required=True)
    parser.add_argument("-o", "--outdir", required=True)
    parser.add_argument("--include-checkpoints", action="store_true")
    parser.add_argument("--summary-name", default="bnd_annotated_by_tool.summary.tsv")
    args = parser.parse_args()

    outdir = Path(args.outdir)
    files = collect_vcf_files(args.input, args.include_checkpoints)
    if not files:
        raise SystemExit("No VCF/VCF.gz files found.")

    results = []
    for infile in files:
        results.append(process_vcf(infile, outdir))

    summary = outdir / args.summary_name
    summary.parent.mkdir(parents=True, exist_ok=True)
    with open(summary, "wt") as out:
        out.write("tool\tinput_vcf\toutput_vcf\tmetric\tcount\n")
        for tool, infile, outfile, counts in results:
            for metric, count in sorted(counts.items()):
                out.write(f"{tool}\t{infile}\t{outfile}\t{metric}\t{count}\n")

    print(f"[DONE] processed_vcfs={len(results)}")
    print(f"[OUT] {outdir}")
    print(f"[SUMMARY] {summary}")


if __name__ == "__main__":
    main()

遍历输入目录（或文件）
  ↓
对每个 VCF：
  1. 读取 header 和 records
  2. 对每条 record：
     - 不是 BND → 原样保留
     - 是 BND   → 调用 annotate_bnd_info 加注释
  3. 写出新 VCF（header 补 BNDINF 定义，records 逐条写）

# 二、处理步骤

### 运行脚本：

In [ ]:
BASE=/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/sv_tools_results

python /mnt/home/ygjx/chenkejin/bnd_format_check/annotate_bnd_with_inferred_svtype.py \
  -i $BASE/cue_result \
     $BASE/delly_result \
     $BASE/lumpy_result \
     $BASE/gridss_result \
     $BASE/svaba_result \
     $BASE/manta_result \
  -o /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_annotated

### 汇总每个工具的转换结果：

In [ ]:
awk -F'\t' '
NR>1{
  count[$1"\t"$4]+=$5
}
END{
  for(k in count) print k"\t"count[k]
}' /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_annotated/bnd_annotated_by_tool.summary.tsv \
| sort